# National Instability Event Classifier
## XGBoost + SHAP on Azure Machine Learning

Trains a binary classifier to predict national-level instability events (coups, civil-war onset) from a country-year panel dataset stored in Azure Data Lake Storage, with full MLflow experiment tracking and SHAP-based explainability running on Azure Machine Learning compute.

| Component | Detail |
|---|---|
| Data source | ADLS Gen2 via AML Datastore or direct URL |
| Input formats | CSV / newline-delimited JSON |
| Algorithm | XGBoost binary:logistic |
| Tuning | RandomizedSearchCV + StratifiedKFold |
| Experiment tracking | MLflow — Azure ML integrated |
| Explainability | SHAP TreeExplainer |

### Expected dataset schema

| Column | Description |
|---|---|
| country_code | ISO 3-letter country code |
| year | Calendar year integer |
| instability_event | Binary target — 1=event 0=no event |
| feature columns | Numeric/categorical country-year indicators |

> **Note:** Typical features include GDP per capita, GDP growth, inflation, civil liberties index, Polity score, military expenditure, ethnic fractionalization, conflict history.

### Workflow

1. Load & explore from ADLS
2. EDA
3. Temporal feature engineering (lags, rolling windows, event history)
4. Temporal train/val/test split — no cross-time leakage
5. Class imbalance handling
6. Hyperparameter search
7. Final model with early stopping
8. Test-set evaluation
9. SHAP explainability (beeswarm, bar, waterfall, dependence, heatmap)
10. Log all metadata/metrics/artefacts/model to MLflow

In [ ]:
%%capture
%pip install \
    'azure-ai-ml>=1.12.0' \
    azure-identity \
    azureml-mlflow \
    azureml-fsspec \
    'xgboost>=2.0' \
    'scikit-learn>=1.3' \
    imbalanced-learn \
    'shap>=0.44' \
    'matplotlib>=3.7' \
    seaborn \
    'pandas>=2.0' \
    numpy \
    adlfs \
    fsspec \
    mlflow --quiet

In [ ]:
import os
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for AML compute
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import shap

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score,
    brier_score_loss,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, ManagedIdentityCredential

warnings.filterwarnings('ignore', category=UserWarning)
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
logger = logging.getLogger(__name__)

print(f'XGBoost : {xgb.__version__}')
print(f'SHAP    : {shap.__version__}')
print(f'MLflow  : {mlflow.__version__}')

In [ ]:

CFG = {
    # Azure ML workspace
    'subscription_id':   '<YOUR_SUBSCRIPTION_ID>',
    'resource_group':    '<YOUR_RESOURCE_GROUP>',
    'workspace_name':    '<YOUR_WORKSPACE_NAME>',

    # ADLS — Option A: registered AML datastore
    'use_datastore':     True,
    'datastore_name':    'adls_instability_data',
    'data_path':         'raw/country_year_panel.csv',

    # ADLS — Option B: direct ADLS Gen2 URL (used when use_datastore=False)
    'adls_account_name': '<YOUR_STORAGE_ACCOUNT>',
    'adls_container':    '<YOUR_CONTAINER>',
    'adls_file_path':    'raw/country_year_panel.csv',

    # Data schema
    'file_format':       'csv',          # 'csv' | 'json'
    'target_col':        'instability_event',
    'country_col':       'country_code',
    'year_col':          'year',
    'drop_cols':         ['country_name'],

    # Temporal split
    'test_year_cutoff':  2018,
    'val_year_cutoff':   2015,

    # Feature engineering
    'lag_features':      True,
    'lag_years':         [1, 2, 3],
    'rolling_windows':   [3, 5],
    'numeric_fill':      'median',       # 'median' | 'mean' | 0

    # Class imbalance  —  'none' | 'scale_pos_weight' | 'smote' | 'undersample'
    'imbalance_strategy': 'scale_pos_weight',

    # XGBoost hyperparameter search
    'n_iter_search':          50,
    'cv_folds':               5,
    'early_stopping_rounds':  20,
    'random_state':           42,
    'param_grid': {
        'n_estimators':      [100, 200, 400, 600, 800],
        'max_depth':         [3, 4, 5, 6, 7, 8],
        'learning_rate':     [0.005, 0.01, 0.05, 0.1, 0.2],
        'subsample':         [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree':  [0.5, 0.6, 0.7, 0.8, 1.0],
        'colsample_bylevel': [0.5, 0.7, 1.0],
        'min_child_weight':  [1, 3, 5, 7, 10],
        'gamma':             [0, 0.1, 0.2, 0.5, 1.0],
        'reg_alpha':         [0, 0.01, 0.1, 0.5, 1.0],
        'reg_lambda':        [0.5, 1.0, 2.0, 5.0],
    },

    # MLflow
    'experiment_name': 'xgboost-instability-classifier',
    'run_tags': {
        'project': 'national-instability',
        'model':   'xgboost',
        'task':    'binary-classification',
    },

    # SHAP
    'shap_sample_size': 500,
    'shap_top_n':       20,

    # Output paths
    'output_dir': 'outputs',
}

OUTPUT_DIR = Path(CFG['output_dir'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'shap').mkdir(exist_ok=True)
(OUTPUT_DIR / 'metrics').mkdir(exist_ok=True)
print('Configuration loaded.')
print(f"Experiment : {CFG['experiment_name']}")
print(f"Test cutoff: years >= {CFG['test_year_cutoff']}")

In [ ]:
try:
    credential = ManagedIdentityCredential()
    ml_client = MLClient(
        credential,
        CFG['subscription_id'],
        CFG['resource_group_name'],
        CFG['workspace_name'],
    )
    ws = ml_client.workspaces.get(CFG['workspace_name'])
    print('Connected to AML workspace via ManagedIdentityCredential')
except Exception:
    credential = DefaultAzureCredential()
    ml_client = MLClient(
        credential,
        CFG['subscription_id'],
        CFG['resource_group_name'],
        CFG['workspace_name'],
    )
    ws = ml_client.workspaces.get(CFG['workspace_name'])
    print('Connected to AML workspace via DefaultAzureCredential')

mlflow_tracking_uri = ws.mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment(CFG['experiment_name'])

print(f'MLflow tracking URI : {mlflow_tracking_uri}')
print(f'Experiment name     : {CFG["experiment_name"]}')

In [ ]:

def load_from_datastore(ml_client, datastore_name, data_path, file_format):
    from azureml.fsspec import AzureMachineLearningFileSystem
    sub  = ml_client._subscription_id
    rg   = ml_client._resource_group_name
    ws   = ml_client.workspace_name
    base = (f'azureml://subscriptions/{sub}/resourcegroups/{rg}'
            f'/workspaces/{ws}/datastores/{datastore_name}/paths/')
    fs = AzureMachineLearningFileSystem(base)
    with fs.open(data_path, 'rb') as f:
        if file_format == 'csv':
            return pd.read_csv(f)
        elif file_format == 'json':
            return pd.read_json(f, orient='records', lines=True)
        else:
            raise ValueError(f'Unsupported file_format: {file_format}')


def load_from_adls_direct(account_name, container, file_path, file_format, credential=None):
    import adlfs, os
    cred = credential or os.environ.get('ADLS_SAS_TOKEN') or os.environ.get('ADLS_ACCOUNT_KEY')
    fs = adlfs.AzureBlobFileSystem(account_name=account_name, credential=cred)
    with fs.open(f'{container}/{file_path}', 'rb') as f:
        if file_format == 'csv':
            return pd.read_csv(f)
        elif file_format == 'json':
            return pd.read_json(f, orient='records', lines=True)
        else:
            raise ValueError(f'Unsupported file_format: {file_format}')

In [ ]:

logger.info('Loading data from ADLS ...')
if CFG['use_datastore']:
    df_raw = load_from_datastore(
        ml_client,
        datastore_name=CFG['datastore_name'],
        data_path=CFG['data_path'],
        file_format=CFG['file_format'],
    )
else:
    df_raw = load_from_adls_direct(
        account_name=CFG['adls_account_name'],
        container=CFG['adls_container'],
        file_path=CFG['adls_file_path'],
        file_format=CFG['file_format'],
    )
logger.info(f'Raw data shape: {df_raw.shape}')
df_raw.head()

In [ ]:

print('=== Shape ===')
print(df_raw.shape)

print('=== dtypes ===')
print(df_raw.dtypes.to_string())

print('=== Missing values (%) ===')
missing = df_raw.isnull().mean().mul(100)
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print('No missing values.')
else:
    print(missing.to_string())

print('=== Target distribution ===')
target_col = CFG['target_col']
print(df_raw[target_col].value_counts().to_string())
rate = df_raw[target_col].mean()
print(f'Positive rate: {rate:.2%}')

year_col = CFG['year_col']
print(f'Year range: {df_raw[year_col].min()} – {df_raw[year_col].max()}')

country_col = CFG['country_col']
print(f'Unique countries: {df_raw[country_col].nunique()}')

rate_by_year = df_raw.groupby(CFG['year_col'])[CFG['target_col']].mean()
fig, ax = plt.subplots(figsize=(12, 4))
rate_by_year.plot(kind='bar', ax=ax)
ax.set_xlabel('Year')
ax.set_ylabel('Instability event rate')
ax.set_title('Annual instability event rate')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'metrics' / 'event_rate_by_year.png', dpi=150)
plt.show()
plt.close()

In [ ]:

def _years_since_last(grp, year_col, target_col):
    grp = grp.sort_values(year_col)
    last_event_year = None
    values = []
    for _, row in grp.iterrows():
        if last_event_year is None:
            values.append(float('nan'))
        else:
            values.append(row[year_col] - last_event_year)
        if row[target_col] == 1:
            last_event_year = row[year_col]
    return pd.Series(values, index=grp.index)


def engineer_features(df, cfg):
    country_col = cfg['country_col']
    year_col    = cfg['year_col']
    target_col  = cfg['target_col']

    df = df.copy()
    df = df.sort_values([country_col, year_col])

    cols_to_drop = [c for c in cfg['drop_cols'] if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    exclude = {country_col, year_col, target_col}
    feature_cols = [c for c in df.select_dtypes(include='number').columns if c not in exclude]

    if cfg['lag_features']:
        for col in feature_cols:
            for lag in cfg['lag_years']:
                df[f'{col}_lag{lag}'] = df.groupby(country_col)[col].shift(lag)
        for col in feature_cols:
            for window in cfg['rolling_windows']:
                df[f'{col}_roll{window}mean'] = (
                    df.groupby(country_col)[col]
                    .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
                )

    df['hist_instability_count'] = (
        df.groupby(country_col)[target_col]
        .transform(lambda s: s.shift(1).expanding().sum())
        .fillna(0)
    )

    df['years_since_last_event'] = (
        df.groupby(country_col, group_keys=False)
        .apply(lambda grp: _years_since_last(grp, year_col, target_col))
    )

    numeric_cols = [c for c in df.select_dtypes(include='number').columns if c != target_col]
    fill_strategy = cfg['numeric_fill']
    if fill_strategy == 'median':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    elif fill_strategy == 'mean':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    else:
        df[numeric_cols] = df[numeric_cols].fillna(0)

    le = LabelEncoder()
    for col in df.select_dtypes(include=['object', 'category']).columns:
        if col != country_col:
            df[col] = le.fit_transform(df[col].astype(str))

    return df


df = engineer_features(df_raw, CFG)
logger.info(f'Engineered shape: {df.shape}')
logger.info(f'Total features: {df.shape[1] - 3}')

In [ ]:

# Never shuffle across time — temporal ordering is the cross-validation strategy
def temporal_split(df, cfg):
    country_col = cfg['country_col']
    year_col    = cfg['year_col']
    target_col  = cfg['target_col']

    feature_cols = [c for c in df.columns if c not in (country_col, year_col, target_col)]

    train_mask = df[year_col] < cfg['val_year_cutoff']
    val_mask   = (df[year_col] >= cfg['val_year_cutoff']) & (df[year_col] < cfg['test_year_cutoff'])
    test_mask  = df[year_col] >= cfg['test_year_cutoff']

    X_train = df.loc[train_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    X_val   = df.loc[val_mask,   feature_cols]
    y_val   = df.loc[val_mask,   target_col]
    X_test  = df.loc[test_mask,  feature_cols]
    y_test  = df.loc[test_mask,  target_col]

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = temporal_split(df, CFG)

feature_names = X_train.columns.tolist()

print(f'{"Split":<10} {"Rows":>8} {"Pos rate":>10}')
print('-' * 30)
print(f'{"Train":<10} {len(X_train):>8} {y_train.mean():>10.2%}')
print(f'{"Val":<10} {len(X_val):>8} {y_val.mean():>10.2%}')
print(f'{"Test":<10} {len(X_test):>8} {y_test.mean():>10.2%}')

In [ ]:

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / max(pos_count, 1)
logger.info(f'Class ratio (neg/pos) in train: {spw:.1f}')

strategy = CFG['imbalance_strategy']
if strategy == 'smote':
    k = min(5, pos_count - 1)
    sampler = SMOTE(random_state=CFG['random_state'], k_neighbors=k)
    X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)
    logger.info(f'After SMOTE: {len(X_train_res)} rows')
elif strategy == 'undersample':
    sampler = RandomUnderSampler(random_state=CFG['random_state'])
    X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)
    logger.info(f'After undersampling: {len(X_train_res)} rows')
else:
    X_train_res, y_train_res = X_train, y_train
    logger.info(f'No resampling — using scale_pos_weight={spw:.1f} in XGBoost')

print(f'Training set for model: {len(X_train_res)} rows')

In [ ]:

# Disable autolog during search to keep experiment clean
mlflow.xgboost.autolog(disable=True)

base_params = {
    'objective':         'binary:logistic',
    'eval_metric':       'aucpr',
    'tree_method':       'hist',
    'use_label_encoder': False,
    'random_state':      CFG['random_state'],
}
if CFG['imbalance_strategy'] == 'scale_pos_weight':
    base_params['scale_pos_weight'] = spw

# Search on train + val combined
X_search = pd.concat([X_train_res, X_val], ignore_index=True)
y_search = pd.concat([y_train_res, y_val], ignore_index=True)

cv = StratifiedKFold(n_splits=CFG['cv_folds'], shuffle=False)
search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(**base_params),
    param_distributions=CFG['param_grid'],
    n_iter=CFG['n_iter_search'],
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=False,
    verbose=1,
    random_state=CFG['random_state'],
)
logger.info(f"Starting RandomizedSearchCV ({CFG['n_iter_search']} iterations, {CFG['cv_folds']}-fold CV) ...")
search.fit(X_search, y_search)

best_params = {**base_params, **search.best_params_}
logger.info(f'Best CV ROC-AUC : {search.best_score_:.4f}')
logger.info(f'Best params     : {search.best_params_}')

In [ ]:

mlflow.xgboost.autolog(log_input_examples=True, log_model_signatures=True, silent=True)
active_run = mlflow.start_run(tags=CFG['run_tags'])
RUN_ID = active_run.info.run_id
logger.info(f'MLflow run ID: {RUN_ID}')

# Log all scalar CFG params
for k, v in CFG.items():
    if not isinstance(v, (dict, list)):
        mlflow.log_param(k, str(v))

# Log best hyperparameters
for k, v in search.best_params_.items():
    mlflow.log_param(f'best_{k}', v)

mlflow.log_metric('cv_best_roc_auc', search.best_score_)
mlflow.log_param('n_features', len(feature_names))
mlflow.log_param('train_size', int(len(X_train_res)))
mlflow.log_param('test_size', int(len(X_test)))
mlflow.log_param('positive_rate_train', float(y_train_res.mean()))
mlflow.log_param('positive_rate_test', float(y_test.mean()))
mlflow.log_param('class_ratio_neg_pos', float(spw))
print(f'Run {RUN_ID} started.')

In [ ]:

# Combine train+val; use last 10% as early-stopping monitor set
X_trainval = pd.concat([X_train_res, X_val], ignore_index=True)
y_trainval = pd.concat([y_train_res, y_val], ignore_index=True)
split_idx = int(len(X_trainval) * 0.9)
X_fit, y_fit = X_trainval.iloc[:split_idx], y_trainval.iloc[:split_idx]
X_es,  y_es  = X_trainval.iloc[split_idx:], y_trainval.iloc[split_idx:]

final_params = {**best_params, 'n_estimators': 1000}
final_model = xgb.XGBClassifier(**final_params)
final_model.fit(
    X_fit, y_fit,
    eval_set=[(X_es, y_es)],
    early_stopping_rounds=CFG['early_stopping_rounds'],
    verbose=False,
)
best_iteration = final_model.best_iteration
logger.info(f'Best iteration (early stopping): {best_iteration}')
mlflow.log_param('best_iteration', best_iteration)

y_pred_proba = final_model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= 0.5).astype(int)

test_metrics = {
    'test_roc_auc':       roc_auc_score(y_test, y_pred_proba),
    'test_avg_precision': average_precision_score(y_test, y_pred_proba),
    'test_f1':            f1_score(y_test, y_pred, zero_division=0),
    'test_brier_score':   brier_score_loss(y_test, y_pred_proba),
}
mlflow.log_metrics(test_metrics)
logger.info('Test metrics: ' + ', '.join(f'{k}={v:.4f}' for k, v in test_metrics.items()))

In [ ]:

print('=== Test-set classification report ===')
print(classification_report(y_test, y_pred, target_names=['stable', 'instability']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['stable', 'instability'],
    ax=ax, colorbar=False,
)
ax.set_title('Confusion Matrix — test set')
plt.tight_layout()
cm_path = OUTPUT_DIR / 'metrics' / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()
mlflow.log_artifact(str(cm_path), artifact_path='metrics')

# ROC + PR curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='XGBoost')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_title('ROC Curve — test set')
PrecisionRecallDisplay.from_predictions(y_test, y_pred_proba, ax=axes[1], name='XGBoost')
axes[1].set_title('Precision-Recall Curve — test set')
plt.tight_layout()
roc_path = OUTPUT_DIR / 'metrics' / 'roc_pr_curves.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.close()
mlflow.log_artifact(str(roc_path), artifact_path='metrics')
print('Evaluation plots saved and logged to MLflow.')

## SHAP Explainability

[SHAP (SHapley Additive exPlanations)](https://shap.readthedocs.io) quantifies each feature's contribution to individual predictions by fairly distributing the difference between the model output and a baseline across all input features.

`TreeExplainer` is exact and fast for tree-based models like XGBoost — it runs in polynomial time by exploiting the tree structure rather than sampling.

### Plot types produced

| Plot | What it shows |
|---|---|
| Beeswarm | Per-sample SHAP impact ranked by mean \|SHAP\|; each dot is one sample, colour = feature value |
| Bar | Global mean \|SHAP\| importance — single number per feature summarising overall contribution |
| Waterfall | Single-prediction breakdown — cumulative feature contributions from baseline to final output |
| Scatter / Dependence | Feature value vs SHAP value with interaction colouring — reveals non-linearity and interactions |
| Heatmap | Sample × feature SHAP matrix ordered by model output — reveals clusters of similar explanations |

> **Sign convention:** positive SHAP value = pushes the prediction toward instability (class 1); negative SHAP value = pushes toward stability (class 0).

In [ ]:

shap_path = OUTPUT_DIR / 'shap'

explainer = shap.TreeExplainer(final_model)

# Sample test rows — mix high-risk and low-risk for richer visualisations
sample_n = min(CFG['shap_sample_size'], len(X_test))
sorted_idx = np.argsort(y_pred_proba)
half = sample_n // 2
shap_row_idx = np.concatenate([sorted_idx[-half:], sorted_idx[:half]])
shap_row_idx = np.unique(shap_row_idx)[:sample_n]

X_shap = X_test.iloc[shap_row_idx].reset_index(drop=True)
y_shap_pred = y_pred_proba[shap_row_idx]

shap_explanation = explainer(X_shap)
print(f'SHAP values shape : {shap_explanation.values.shape}')

mean_abs_shap = np.abs(shap_explanation.values).mean(axis=0)
top_features = (
    pd.Series(mean_abs_shap, index=feature_names[:len(mean_abs_shap)])
    .sort_values(ascending=False)
)
print('Top 10 features by mean |SHAP|:')
print(top_features.head(10).to_string())

In [ ]:

plt.figure()
shap.plots.beeswarm(shap_explanation, max_display=CFG['shap_top_n'], show=False)
plt.tight_layout()
beeswarm_path = shap_path / 'shap_beeswarm.png'
plt.savefig(beeswarm_path, dpi=150, bbox_inches='tight')
plt.close()
mlflow.log_artifact(str(beeswarm_path), artifact_path='shap')
print(f'Saved: {beeswarm_path}')

In [ ]:

plt.figure()
shap.plots.bar(shap_explanation, max_display=CFG['shap_top_n'], show=False)
plt.tight_layout()
bar_path = shap_path / 'shap_bar.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.close()
mlflow.log_artifact(str(bar_path), artifact_path='shap')
print(f'Saved: {bar_path}')

In [ ]:

idx_high = int(np.argmax(y_shap_pred))
idx_low  = int(np.argmin(y_shap_pred))

for label, idx in [('highest_risk', idx_high), ('lowest_risk', idx_low)]:
    plt.figure()
    shap.plots.waterfall(shap_explanation[idx], show=False)
    plt.tight_layout()
    wf_path = shap_path / f'shap_waterfall_{label}.png'
    plt.savefig(wf_path, dpi=150, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(str(wf_path), artifact_path='shap')
    print(f'Saved: {wf_path}')

In [ ]:

top3_features = top_features.head(3).index.tolist()
for feat in top3_features:
    if feat not in X_shap.columns:
        continue
    plt.figure()
    shap.plots.scatter(shap_explanation[:, feat], show=False)
    plt.tight_layout()
    safe_name = feat.replace('/', '_').replace(' ', '_')
    dep_path = shap_path / f'shap_dependence_{safe_name}.png'
    plt.savefig(dep_path, dpi=150, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(str(dep_path), artifact_path='shap')
    print(f'Saved: {dep_path}')

In [ ]:

plt.figure()
shap.plots.heatmap(shap_explanation, max_display=CFG['shap_top_n'], show=False)
plt.tight_layout()
heatmap_path = shap_path / 'shap_heatmap.png'
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.close()
mlflow.log_artifact(str(heatmap_path), artifact_path='shap')
print(f'Saved: {heatmap_path}')

In [ ]:

signature = infer_signature(X_test, y_pred_proba)
mlflow.xgboost.log_model(
    final_model,
    artifact_path='model',
    signature=signature,
    input_example=X_test.head(3),
)
mlflow.end_run()
print(f'Run {RUN_ID} complete.')
print(f"View in Azure ML Studio > Experiments > {CFG['experiment_name']}")
print('Artefacts logged: model, metrics plots, SHAP visualisations')

## Summary

| Item | Value |
|---|---|
| MLflow run ID | See `RUN_ID` variable above |
| Experiment | `xgboost-instability-classifier` |
| Model artefact | `outputs/model/` |
| Evaluation plots | `outputs/metrics/` |
| SHAP plots | `outputs/shap/` |

### Interpreting SHAP plots
- **Beeswarm**: each dot is one country-year. Red = high feature value, blue = low. Dots right of centre push toward instability.
- **Bar**: average absolute SHAP — global feature importance ranking.
- **Waterfall**: breaks down a single prediction into feature contributions.
- **Dependence**: shows how a feature's value relates to its SHAP value; colour = interaction feature.
- **Heatmap**: rows = features, columns = samples ordered by model output; reveals clusters.

### Suggested next steps
1. Adjust `CFG['test_year_cutoff']` to evaluate on different temporal windows.
2. Use SHAP-ranked features to prune low-importance engineered features.
3. Tune the classification threshold with `PrecisionRecallDisplay` to balance recall vs precision for your use case.
4. Register the model in Azure ML Model Registry via `ml_client.models.create_or_update(...)`.
5. Consider temporal cross-validation (`TimeSeriesSplit`) for a more rigorous evaluation.